# HW2 — Data Cleaning and Validation

## 1.1 Load and Inspect the Data

In [1]:
from pathlib import Path

Path("data/data.db").is_file()

True

In [2]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("data/data.db")
raw_prices = pd.read_sql_query("SELECT * FROM ohlc_hw", connection)
connection.close()

raw_prices.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [3]:
raw_prices.info()
raw_prices.describe()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 24.3 MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [4]:
dates = pd.to_datetime(raw_prices["ts"])
ticker_labels = sorted(raw_prices["ticker"].dropna().unique())

print(f"Date range: {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")
print(f"Distinct raw ticker labels: {len(ticker_labels)}")
print(ticker_labels)

Date range: 1962-01-02 to 2024-12-06
Distinct raw ticker labels: 150
['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span></a>', '<a target="_blank"><span style="color: blue">J

### What the columns mean

- `ts` is the trading date.
- `ticker` is the stock symbol.
- `open` and `close` are the opening and closing prices.
- `high` and `low` are the highest and lowest prices during the trading session.
- `adj_close` is the closing price adjusted for events such as stock splits and dividends.
- `volume` is the number of shares traded during the session.

## 1.2 Detect Data Quality Issues

In [5]:
prices = raw_prices.copy()
prices["ticker_original"] = prices["ticker"]

prices["ticker"] = (
    prices["ticker_original"]
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
    .str.upper()
)

tickers = sorted(prices["ticker"].dropna().unique())
print(f"Distinct standardized tickers: {len(tickers)}")
print(tickers)

Distinct standardized tickers: 30
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


In [6]:
record_columns = raw_prices.columns.tolist()

exact_duplicate = prices.duplicated(subset=record_columns)
duplicate_rows = prices.loc[exact_duplicate].copy()
unique_prices = prices.loc[~exact_duplicate].copy()

same_stock_and_date = unique_prices.duplicated(
    subset=["ticker", "ts"], keep=False
)
conflicting_rows = unique_prices.loc[same_stock_and_date].copy()

print(f"Extra copies of exact records: {len(duplicate_rows):,}")
print(f"Rows with conflicting stock/date records: {len(conflicting_rows):,}")

Extra copies of exact records: 29,460
Rows with conflicting stock/date records: 0


In [7]:
def count_missing_values(data):
    return data.isna().sum()


missing_counts = count_missing_values(unique_prices[record_columns])
missing_counts.to_frame("missing_values")

,missing_values
ts,0
open,0
high,0
low,0
close,0
adj_close,18374
volume,0
ticker,0


In [8]:
connection = sqlite3.connect("data/data.db")
reference_dates = pd.read_sql_query(
    "SELECT DISTINCT ticker, ts FROM ohlc", connection
)
connection.close()


def find_missing_dates(expected, observed):
    keys = ["ticker", "ts"]
    comparison = expected[keys].merge(
        observed[keys], on=keys, how="left", indicator=True
    )
    return comparison.loc[comparison["_merge"] == "left_only", keys]


missing_dates = find_missing_dates(reference_dates, unique_prices)
print(f"Missing trading dates: {len(missing_dates):,}")

Missing trading dates: 0


In [9]:
example_expected = pd.DataFrame({
    "ticker": ["AAPL", "AAPL", "AAPL"],
    "ts": ["2024-06-03", "2024-06-04", "2024-06-05"],
})

# Remove the middle date from the example observations.
example_observed = example_expected.drop(index=1)
example_missing = find_missing_dates(example_expected, example_observed)

example_missing

,ticker,ts
1,AAPL,2024-06-04


In [10]:
gap_summary = reference_dates.groupby("ticker")["ts"].agg(
    start_date="min",
    end_date="max",
)

missing_by_ticker = missing_dates.groupby("ticker").size()
gap_summary["missing_days"] = missing_by_ticker.reindex(
    gap_summary.index, fill_value=0
)

gap_summary

,start_date,end_date,missing_days
ticker,,,
AAPL,1980-12-12,2024-10-18,0
AMGN,1983-06-17,2024-10-18,0
AMZN,1997-05-15,2024-12-06,0
AXP,1972-04-03,2024-10-18,0
BA,1962-01-02,2024-10-18,0
CAT,1962-01-02,2024-10-18,0
CRM,2004-06-23,2024-10-18,0
CSCO,1990-02-16,2024-10-18,0
CVX,1962-01-02,2024-12-06,0


In [11]:
import numpy as np


def flag_return_outliers(data):
    result = data.sort_values(["ticker", "ts"]).copy()

    close = result["close"]
    valid_close = close.where(np.isfinite(close) & (close > 0))
    previous_close = valid_close.groupby(result["ticker"]).shift(1)

    daily_return = (valid_close - previous_close) / previous_close
    result["daily_return"] = daily_return.where(np.isfinite(daily_return))

    by_ticker = result.groupby("ticker")["daily_return"]
    mean_return = by_ticker.transform("mean")
    std_return = by_ticker.transform("std", ddof=1)

    distance = (result["daily_return"] - mean_return).abs()
    result["return_outlier"] = (std_return > 0) & (distance > 3 * std_return)
    return result


return_checks = flag_return_outliers(unique_prices)
return_outliers = return_checks.loc[
    return_checks["return_outlier"], ["ticker", "ts", "daily_return"]
]

print(f"Returns flagged for review: {len(return_outliers):,}")
return_outliers.head()

Returns flagged for review: 5,056


,ticker,ts,daily_return
393432,AAPL,1980-12-26,0.092288
139446,AAPL,1981-04-02,0.087659
107726,AAPL,1981-09-25,-0.129776
176299,AAPL,1982-03-19,0.090173
154954,AAPL,1982-11-22,-0.089089


In [12]:
def check_price_and_volume(data):
    price_columns = ["open", "high", "low", "close", "adj_close"]
    price_values = data[price_columns]
    invalid_prices = price_values.notna() & (
        (price_values <= 0) | ~np.isfinite(price_values)
    )

    return pd.DataFrame({
        "low_above_high": data["low"] > data["high"],
        "open_outside_range": (
            (data["open"] < data["low"]) | (data["open"] > data["high"])
        ),
        "close_outside_range": (
            (data["close"] < data["low"]) | (data["close"] > data["high"])
        ),
        "invalid_price": invalid_prices.any(axis=1),
        "zero_volume": data["volume"] == 0,
    })


price_volume_checks = check_price_and_volume(unique_prices)
price_volume_checks.sum().to_frame("rows_flagged")

,rows_flagged
low_above_high,0
open_outside_range,5
close_outside_range,0
invalid_price,5
zero_volume,75


In [13]:
def issue_rows(rows, issue_type, description, severity):
    report_rows = rows[["ticker", "ts"]].rename(columns={"ts": "issue_date"})
    report_rows["issue_type"] = issue_type
    report_rows["description"] = description
    report_rows["severity"] = severity
    return report_rows[["ticker", "issue_type", "issue_date", "description", "severity"]]


report_parts = [
    issue_rows(duplicate_rows, "exact_duplicate", "Extra copy of a record.", "low"),
    issue_rows(conflicting_rows, "conflicting_record", "Records disagree for this stock and date.", "high"),
    issue_rows(missing_dates, "missing_date", "Expected trading day is absent.", "high"),
    issue_rows(return_outliers, "return_outlier", "Return is more than 3 standard deviations from the stock's mean.", "medium"),
]

for column in record_columns:
    missing_rows = unique_prices.loc[unique_prices[column].isna()]
    report_parts.append(
        issue_rows(missing_rows, "missing_value", f"{column} is missing.", "high")
    )

check_details = {
    "low_above_high": ("The low is above the high.", "high"),
    "open_outside_range": ("The open is outside the daily range.", "high"),
    "close_outside_range": ("The close is outside the daily range.", "high"),
    "invalid_price": ("At least one price is nonpositive or nonfinite.", "high"),
    "zero_volume": ("Reported volume is zero.", "low"),
}

for check, (description, severity) in check_details.items():
    flagged_rows = unique_prices.loc[price_volume_checks[check]]
    report_parts.append(issue_rows(flagged_rows, check, description, severity))

quality_report = pd.concat(report_parts, ignore_index=True)
display(quality_report["issue_type"].value_counts().to_frame("issues"))
quality_report.head()

,issues
issue_type,
exact_duplicate,29460
missing_value,18374
return_outlier,5056
zero_volume,75
open_outside_range,5
invalid_price,5


,ticker,issue_type,issue_date,description,severity
0,DIS,exact_duplicate,2004-04-28,Extra copy of a record.,low
1,CAT,exact_duplicate,1962-10-16,Extra copy of a record.,low
2,TRV,exact_duplicate,1975-06-02,Extra copy of a record.,low
3,MRK,exact_duplicate,2018-09-21,Extra copy of a record.,low
4,NVDA,exact_duplicate,2021-07-21,Extra copy of a record.,low


## 1.3 Clean the Data

In [14]:
price_checks = [
    "low_above_high",
    "open_outside_range",
    "close_outside_range",
    "invalid_price",
]
has_price_issue = price_volume_checks[price_checks].any(axis=1)

unique_prices.loc[
    has_price_issue,
    ["ticker", "ts", "open", "high", "low", "close", "adj_close"],
].sort_values(["ticker", "ts"])

,ticker,ts,open,high,low,close,adj_close
287343,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750
350808,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012
224790,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090
364568,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447
99637,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300


In [15]:
raw_price_columns = ["open", "high", "low", "close"]


def fill_raw_prices(data):
    work = data.sort_values(["ticker", "ts"]).copy()

    all_missing = work[raw_price_columns].isna().all(axis=1)
    dropped = work.loc[all_missing].copy()
    work = work.loc[~all_missing].copy()

    raw = work[raw_price_columns]
    usable = raw.where(np.isfinite(raw) & (raw > 0))
    filled = usable.groupby(work["ticker"]).ffill()

    filled_cells = usable.isna() & filled.notna()
    work["raw_price_filled"] = filled_cells.any(axis=1)
    work["close_filled"] = filled_cells["close"]
    work[raw_price_columns] = filled
    return work, dropped


working_prices, dropped_all_missing = fill_raw_prices(unique_prices)

print(f"Rows dropped because all four raw prices were missing: {len(dropped_all_missing)}")
working_prices[raw_price_columns].isna().sum()

Rows dropped because all four raw prices were missing: 0


open     5
high     0
low      0
close    0
dtype: int64

In [16]:
def split_raw_price_rows(data):
    missing_raw = data[raw_price_columns].isna().any(axis=1)
    checks = check_price_and_volume(data)
    bad_range = checks[
        ["low_above_high", "open_outside_range", "close_outside_range"]
    ].any(axis=1)

    needs_review = missing_raw | bad_range
    quarantined = data.loc[needs_review].copy()
    quarantined["quarantine_reason"] = np.where(
        missing_raw.loc[needs_review],
        "Raw price still missing after forward fill",
        "Raw prices violate the daily range after forward fill",
    )

    return data.loc[~needs_review].copy(), quarantined


valid_raw_prices, quarantined_raw = split_raw_price_rows(working_prices)

print(f"Rows with valid raw prices: {len(valid_raw_prices):,}")
quarantined_raw[["ticker", "ts", "quarantine_reason"]]

Rows with valid raw prices: 368,246


,ticker,ts,quarantine_reason
287343,CVX,1962-01-02,Raw price still missing after forward fill
350808,JNJ,1962-01-02,Raw price still missing after forward fill
224790,MCD,1966-07-05,Raw price still missing after forward fill
364568,MRK,1962-01-02,Raw price still missing after forward fill
99637,PG,1962-01-02,Raw price still missing after forward fill


In [17]:
connection = sqlite3.connect("data/data.db")
reference_prices = pd.read_sql_query(
    "SELECT ticker, ts, close, adj_close FROM ohlc", connection
)
connection.close()

reference_close = reference_prices["close"].where(reference_prices["close"] > 0)
factor = reference_prices["adj_close"] / reference_close
reference_prices["adjustment_factor"] = factor.where(
    np.isfinite(factor) & (factor > 0)
)

adjusted_prices = valid_raw_prices.merge(
    reference_prices[["ticker", "ts", "adjustment_factor"]],
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
)

adj_close = adjusted_prices["adj_close"]
needs_adjustment = (
    ~np.isfinite(adj_close) | (adj_close <= 0) | adjusted_prices["close_filled"]
)

reconstructed = adjusted_prices["close"] * adjusted_prices["adjustment_factor"]
reconstructed = reconstructed.where(np.isfinite(reconstructed) & (reconstructed > 0))

adjusted_prices["adj_close"] = adj_close.where(~needs_adjustment, reconstructed)
adjusted_prices["adj_close_repaired"] = needs_adjustment & reconstructed.notna()

print(f"Adjusted closes repaired: {adjusted_prices['adj_close_repaired'].sum():,}")
print(f"Adjusted closes still unresolved: {adjusted_prices['adj_close'].isna().sum():,}")

Adjusted closes repaired: 18,374
Adjusted closes still unresolved: 0


In [18]:
has_reference_factor = adjusted_prices["adjustment_factor"].notna()
matches_reference = np.isclose(
    adjusted_prices["adj_close"],
    reconstructed,
    rtol=1e-6,
    atol=1e-8,
)

mismatch = has_reference_factor & ~matches_reference
adjustment_mismatches = adjusted_prices.loc[
    mismatch, ["ticker", "ts", "close", "adj_close", "adjustment_factor"]
].copy()
adjustment_mismatches["expected_adj_close"] = reconstructed.loc[mismatch]

print(f"Rows without a usable reference factor: {(~has_reference_factor).sum():,}")
print(f"Adjusted closes that differ from the reference calculation: {len(adjustment_mismatches):,}")
adjustment_mismatches.head()

Rows without a usable reference factor: 0
Adjusted closes that differ from the reference calculation: 10,417


,ticker,ts,close,adj_close,adjustment_factor,expected_adj_close
35,AAPL,1981-02-03,0.12333,1.264587,0.770853,0.095069
93,AAPL,1981-04-28,0.12612,2.036653,0.770854,0.097220
101,AAPL,1981-05-08,0.12500,2.129290,0.770878,0.096360
157,AAPL,1981-07-29,0.10603,0.506329,0.770857,0.081734
163,AAPL,1981-08-06,0.11272,2.009828,0.770899,0.086896


In [19]:
reference_closes = reference_prices[["ticker", "ts", "close"]].rename(
    columns={"close": "reference_close"}
)

adjustment_review = adjustment_mismatches.merge(
    reference_closes, on=["ticker", "ts"], how="left", validate="one_to_one"
)

adjustment_review["raw_close_matches"] = np.isclose(
    adjustment_review["close"],
    adjustment_review["reference_close"],
    rtol=1e-6,
    atol=1e-8,
)

print(f"Mismatches with matching raw closes: {adjustment_review['raw_close_matches'].sum():,}")
adjustment_review[
    ["ticker", "ts", "close", "reference_close", "adj_close", "expected_adj_close"]
].head()

Mismatches with matching raw closes: 10,417


,ticker,ts,close,reference_close,adj_close,expected_adj_close
0,AAPL,1981-02-03,0.12333,0.12333,1.264587,0.095069
1,AAPL,1981-04-28,0.12612,0.12612,2.036653,0.097220
2,AAPL,1981-05-08,0.12500,0.12500,2.129290,0.096360
3,AAPL,1981-07-29,0.10603,0.10603,0.506329,0.081734
4,AAPL,1981-08-06,0.11272,0.11272,2.009828,0.086896


In [20]:
assert adjustment_review["raw_close_matches"].all(), (
    "Review rows where the raw close also differs before correcting them."
)

consistent_prices = adjusted_prices.copy()
corrected = mismatch & reconstructed.notna()

consistent_prices.loc[corrected, "adj_close"] = reconstructed.loc[corrected]
consistent_prices["adj_close_corrected"] = corrected
consistent_prices["adj_close_repaired"] = (
    adjusted_prices["adj_close_repaired"] | corrected
)

adjustment_issues = issue_rows(
    adjustment_mismatches,
    "inconsistent_adj_close",
    "Adjusted close disagrees with the matched reference factor.",
    "high",
)
quality_report_before_repair = pd.concat(
    [quality_report, adjustment_issues], ignore_index=True
)

print(f"Inconsistent adjusted closes corrected: {corrected.sum():,}")
print(f"Total adjusted closes repaired: {consistent_prices['adj_close_repaired'].sum():,}")

Inconsistent adjusted closes corrected: 10,417
Total adjusted closes repaired: 28,791


In [21]:
reference_returns = flag_return_outliers(reference_prices)[
    ["ticker", "ts", "daily_return"]
].rename(columns={"daily_return": "reference_return"})

outlier_review = return_outliers.merge(
    reference_returns, on=["ticker", "ts"], how="left", validate="one_to_one"
)

outlier_review["matches_reference"] = np.isclose(
    outlier_review["daily_return"],
    outlier_review["reference_return"],
    rtol=1e-6,
    atol=1e-8,
)

print(f"Flagged returns matching the reference: {outlier_review['matches_reference'].sum():,}")
print(f"Flagged returns still to investigate: {(~outlier_review['matches_reference']).sum():,}")

outlier_review.sort_values(
    "daily_return", key=lambda returns: returns.abs(), ascending=False
).head()

Flagged returns matching the reference: 5,056
Flagged returns still to investigate: 0


,ticker,ts,daily_return,reference_return,matches_reference
4429,TRV,1981-08-10,1.000000,1.000000,True
4425,TRV,1976-10-28,1.000000,1.000000,True
4427,TRV,1978-12-20,0.997512,0.997512,True
3066,MCD,1968-01-02,-0.753620,-0.753620,True
98,AAPL,2000-09-29,-0.518695,-0.518695,True


In [22]:
def fill_missing_volume(data):
    result = data.copy()
    result["volume_filled"] = result["volume"].isna()
    result["zero_volume_reported"] = result["volume"].eq(0).fillna(False)
    result["volume"] = result["volume"].fillna(0)
    return result


outlier_flags = return_outliers[["ticker", "ts"]].assign(
    raw_return_outlier=True
)

flagged_prices = consistent_prices.merge(
    outlier_flags, on=["ticker", "ts"], how="left", validate="one_to_one"
)
flagged_prices["raw_return_outlier"] = flagged_prices["raw_return_outlier"].notna()

ready_prices = fill_missing_volume(flagged_prices)

flag_columns = ["raw_return_outlier", "volume_filled", "zero_volume_reported"]
ready_prices[flag_columns].sum().to_frame("rows_flagged")

,rows_flagged
raw_return_outlier,5056
volume_filled,0
zero_volume_reported,74


In [23]:
expected_adj_close = ready_prices["close"] * ready_prices["adjustment_factor"]
bad_adjustment = ~np.isclose(
    ready_prices["adj_close"], expected_adj_close, rtol=1e-6, atol=1e-8
)

quarantined_adjusted = ready_prices.loc[bad_adjustment].copy()
quarantined_adjusted["quarantine_reason"] = "Adjusted close could not be verified"
cleaned_prices = ready_prices.loc[~bad_adjustment].copy()

row_counts = pd.Series({
    "Exact duplicate copies removed": len(duplicate_rows),
    "Rows dropped with all four raw prices missing": len(dropped_all_missing),
    "Rows quarantined for raw price issues": len(quarantined_raw),
    "Rows quarantined for adjusted price issues": len(quarantined_adjusted),
    "Rows retained": len(cleaned_prices),
}, name="rows")

assert row_counts.sum() == len(raw_prices), "Some input rows are unaccounted for."
display(row_counts.to_frame())

quarantined_raw.loc[quarantined_raw["volume"].eq(0), ["ticker", "ts", "volume"]]

,rows
Exact duplicate copies removed,29460
Rows dropped with all four raw prices missing,0
Rows quarantined for raw price issues,5
Rows quarantined for adjusted price issues,0
Rows retained,368246


,ticker,ts,volume
350808,JNJ,1962-01-02,0


In [24]:
final_checks = check_price_and_volume(cleaned_prices)
all_price_columns = raw_price_columns + ["adj_close"]

validation = pd.Series({
    "Rows with a missing price": cleaned_prices[all_price_columns].isna().any(axis=1).sum(),
    "Rows with a nonpositive or nonfinite price": final_checks["invalid_price"].sum(),
    "Rows with invalid daily price bounds": final_checks[
        ["low_above_high", "open_outside_range", "close_outside_range"]
    ].any(axis=1).sum(),
    "Repeated ticker/date pairs": cleaned_prices.duplicated(["ticker", "ts"]).sum(),
    "Rows with invalid volume": (
        ~np.isfinite(cleaned_prices["volume"]) | (cleaned_prices["volume"] < 0)
    ).sum(),
    "Rows with dates outside YYYY-MM-DD format": (
        ~cleaned_prices["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
    ).sum(),
}, name="rows_flagged")

display(validation.to_frame())
assert validation.eq(0).all(), "Resolve the remaining validation issues before saving."

,rows_flagged
Rows with a missing price,0
Rows with a nonpositive or nonfinite price,0
Rows with invalid daily price bounds,0
Repeated ticker/date pairs,0
Rows with invalid volume,0
Rows with dates outside YYYY-MM-DD format,0


In [25]:
numeric_columns = all_price_columns + ["volume"]

before_stats = raw_prices[numeric_columns].describe().T
after_stats = cleaned_prices[numeric_columns].describe().T

before_after = pd.concat(
    {"Before cleaning": before_stats, "After cleaning": after_stats},
    names=["stage", "column"],
)

before_after[["count", "mean", "std", "min", "50%", "max"]]

count          mean           std       min  \
stage           column                                                      
Before cleaning open       397711.0  4.628548e+01  6.713259e+01  0.000000   
                high       397711.0  4.674766e+01  6.777821e+01  0.005208   
                low        397711.0  4.581628e+01  6.647718e+01  0.004801   
                close      397711.0  4.629290e+01  6.714188e+01  0.005208   
                adj_close  377856.0  5.165225e+01  1.945840e+02  0.002945   
                volume     397711.0  3.097773e+07  1.266237e+08  0.000000   
After cleaning  open       368246.0  4.630939e+01  6.714325e+01  0.005208   
                high       368246.0  4.677182e+01  6.778932e+01  0.005208   
                low        368246.0  4.584038e+01  6.648945e+01  0.004801   
                close      368246.0  4.631733e+01  6.715494e+01  0.005208   
                adj_close  368246.0  3.727467e+01  6.358787e+01  0.002945   
                volume     368246.0  3.095184e+07  1.266153e+08  0.000000   

                                  50%           max  
stage           column                               
Before cleaning open            19.85  6.042600e+02  
                high            20.06  6.086300e+02  
                low             19.63  5.978800e+02  
                close           19.86  6.054000e+02  
                adj_close       11.96  1.044251e+04  
                volume     4648200.00  9.276606e+09  
After cleaning  open            19.89  6.042600e+02  
                high            20.09  6.086300e+02  
                low             19.67  5.978800e+02  
                close           19.91  6.054000e+02  
                adj_close       11.46  6.054000e+02  
                volume     4649648.00  9.276606e+09

In [26]:
demo_prices = pd.DataFrame({
    "ticker": ["DEMO"] * 4,
    "ts": ["2024-06-03", "2024-06-04", "2024-06-05", "2024-06-06"],
    "open": [10.0, np.nan, 20.0, np.nan],
    "high": [12.0, 12.0, 22.0, np.nan],
    "low": [9.0, 9.0, 19.0, np.nan],
    "close": [11.0, 11.0, np.nan, np.nan],
    "adj_close": [11.0, 11.0, 21.0, np.nan],
    "volume": [0, np.nan, 100, 100],
})

demo_filled, demo_dropped = fill_raw_prices(demo_prices)
demo_valid, demo_quarantined = split_raw_price_rows(demo_filled)
demo_valid = fill_missing_volume(demo_valid)

display(demo_valid[[
    "ts", "open", "raw_price_filled", "volume", "volume_filled", "zero_volume_reported"
]])
display(demo_quarantined[["ts", "close", "low", "high", "quarantine_reason"]])
print("Dropped dates:", demo_dropped["ts"].tolist())

,ts,open,raw_price_filled,volume,volume_filled,zero_volume_reported
0,2024-06-03,10.0,False,0.0,False,True
1,2024-06-04,10.0,True,0.0,True,False


,ts,close,low,high,quarantine_reason
2,2024-06-05,11.0,19.0,22.0,Raw prices violate the daily range after forwa...


Dropped dates: ['2024-06-06']


In [27]:
from contextlib import closing

with closing(sqlite3.connect("file:data/data.db?mode=rw", uri=True)) as connection:
    cleaned_prices.to_sql(
        "ohlc_hw_cleaned", connection, if_exists="replace", index=False
    )
    saved_count = connection.execute(
        "SELECT COUNT(*) FROM ohlc_hw_cleaned"
    ).fetchone()[0]

assert saved_count == len(cleaned_prices), "The saved row count differs."
print(f"Saved and verified {saved_count:,} rows in ohlc_hw_cleaned.")

Saved and verified 368,246 rows in ohlc_hw_cleaned.


### Cleaning decisions and validation

- **Severity.** Missing values, missing trading dates, conflicting records, and invalid or inconsistent prices are high severity because they can make an observation unusable. Extreme returns are medium severity because they need review. Exact duplicate copies and reported zero volumes are low severity because the original observation can be retained.
- **Raw prices.** I treated zero opens as missing and used earlier values from the same ticker to fill gaps where possible. I quarantined five rows because each was the ticker’s first observation and had no earlier open to carry forward. I checked the daily price bounds again after filling.
- **Adjusted close.** I used the adjustment factor from the matching ticker and date in `ohlc` to reconstruct 18,374 missing values and correct 10,417 inconsistent values. The raw closes on all 10,417 inconsistent rows matched the reference.
- **Returns and volume.** All 5,056 flagged raw-close returns matched returns calculated from `ohlc`, so I kept the observations and their flags. I retained 74 reported zero volumes and kept a separate flag for missing volume filled with zero. No retained row needed a volume fill.
- **Final validation.** After removing 29,460 duplicate copies and quarantining five rows, 368,246 rows passed the validation checks. The SQLite row count matched the cleaned DataFrame.